### Part A

**Question:** Why cannot the raw data be ranked as is?

# Why raw data cannot be ranked as is

**Question:** How does price relate to recorded quantity for comparable products? The starting population is raw rows with `Quantity > 0` and `UnitPrice > 0`. This is a provisional recorded-sales population, not verified demand or completed sales.

In [1]:
from pathlib import Path as Path02
import sys as sys02
import pandas as pandas02
from IPython.display import display as display02

project_root_candidates_02 = (Path02.cwd(), *Path02.cwd().parents)
project_root_02 = next(
    (candidate for candidate in project_root_candidates_02 if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root_02 is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root_02) not in sys02.path:
    sys02.path.insert(0, str(project_root_02))
from src.clean import load_raw as load_raw_02, match_cancelled_orders as match_cancelled_orders_02

relative_data_path_02 = Path02("data") / "raw" / "Online Retail.xlsx"
data_path_02 = project_root_02 / relative_data_path_02
retail_02 = load_raw_02(data_path_02)
retail_snapshot_02 = retail_02.copy(deep=True)
print(f"Workbook: {relative_data_path_02}")
print(f"Raw rows and columns: {retail_02.shape}")

Workbook: data\raw\Online Retail.xlsx
Raw rows and columns: (541909, 8)


## Product rankings

The two rankings use only the starting population. Ties are ordered by `StockCode` for reproducibility.

In [2]:
positive_population_02 = retail_02.loc[
    retail_02["Quantity"].gt(0) & retail_02["UnitPrice"].gt(0)
].copy()
quantity_metrics_02 = positive_population_02.groupby("StockCode", as_index=False).agg(
    total_quantity=("Quantity", "sum")
)
quantity_ranking_02 = quantity_metrics_02.sort_values(
    ["total_quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
top20_quantity_02 = quantity_ranking_02.head(20)
print("Top 20 products by total positive recorded quantity:")
display02(top20_quantity_02)

Top 20 products by total positive recorded quantity:


,StockCode,total_quantity
0,23843,80995
1,23166,78033
2,22197,56921
3,84077,55047
4,85099B,48474
5,85123A,37952
6,84879,36461
7,21212,36419
8,23084,30788
9,22492,26633


In [3]:
invoice_metrics_02 = positive_population_02.groupby("StockCode", as_index=False).agg(
    distinct_invoices=("InvoiceNo", "nunique")
)
invoice_ranking_02 = invoice_metrics_02.sort_values(
    ["distinct_invoices", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
top20_invoices_02 = invoice_ranking_02.head(20)
print("Top 20 products by distinct positive-population invoices:")
display02(top20_invoices_02)

Top 20 products by distinct positive-population invoices:


,StockCode,distinct_invoices
0,85123A,2265
1,85099B,2089
2,22423,1988
3,47566,1685
4,20725,1565
5,84879,1455
6,22197,1392
7,22720,1385
8,21212,1320
9,22383,1284


## Why StockCode 23843 looks suspicious

Inspect every raw row for this code, including its invoice number, quantity, customer, price, and date.

In [4]:
raw_rows_23843_02 = retail_02.loc[
    retail_02["StockCode"].astype("string").str.strip().eq("23843")
].copy()
display02(raw_rows_23843_02)
assert set(raw_rows_23843_02["InvoiceNo"].astype(str)) == {"581483", "C581484"}
assert set(raw_rows_23843_02["Quantity"]) == {80995, -80995}
assert raw_rows_23843_02["CustomerID"].nunique() == 1
print("The raw rows show +80,995 on 581483 and -80,995 on C581484 for the same customer.")

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


The raw rows show +80,995 on 581483 and -80,995 on C581484 for the same customer.


## StockCode 23166: largest line and cancellation

Measure the largest positive-population line against this product's total, then show the exact matched cancellation pair.

In [5]:
stock_codes_23166_02 = retail_02["StockCode"].astype("string").str.strip()
sales_rows_23166_02 = positive_population_02.loc[stock_codes_23166_02.eq("23166")]
total_quantity_23166_02 = int(sales_rows_23166_02["Quantity"].sum())
largest_line_23166_02 = sales_rows_23166_02.sort_values(
    ["Quantity", "InvoiceNo"], ascending=[False, True], kind="mergesort"
).iloc[0]
largest_share_23166_02 = largest_line_23166_02["Quantity"] / total_quantity_23166_02
raw_for_matching_02 = retail_02.copy()
raw_for_matching_02["StockCode"] = raw_for_matching_02["StockCode"].astype(str).str.strip()
matched_positive_02, matched_cancellation_02 = match_cancelled_orders_02(raw_for_matching_02)
matched_pairs_23166_02 = pandas02.DataFrame({
    "InvoiceNo": raw_for_matching_02.loc[matched_positive_02, "InvoiceNo"].to_numpy(),
    "C-InvoiceNo": raw_for_matching_02.loc[matched_cancellation_02, "InvoiceNo"].to_numpy(),
    "StockCode": raw_for_matching_02.loc[matched_positive_02, "StockCode"].to_numpy(),
    "Quantity": raw_for_matching_02.loc[matched_positive_02, "Quantity"].to_numpy(),
    "CustomerID": raw_for_matching_02.loc[matched_positive_02, "CustomerID"].to_numpy(),
})
matched_pairs_23166_02 = matched_pairs_23166_02.loc[
    matched_pairs_23166_02["StockCode"].eq("23166")
]
print(f"23166 positive-population total: {total_quantity_23166_02:,}")
print(
    f"Largest line: {int(largest_line_23166_02['Quantity']):,} of "
    f"{total_quantity_23166_02:,} ({largest_share_23166_02:.1%}); "
    f"InvoiceNo={largest_line_23166_02['InvoiceNo']}"
)
display02(matched_pairs_23166_02.reset_index(drop=True))
assert total_quantity_23166_02 == 78033
assert int(largest_line_23166_02["Quantity"]) == 74215
assert len(matched_pairs_23166_02) >= 1

23166 positive-population total: 78,033
Largest line: 74,215 of 78,033 (95.1%); InvoiceNo=541431


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,541431,C541433,23166,74215,12346.0
1,553607,C557508,23166,240,16684.0
2,560491,C560540,23166,1,12415.0
3,562374,C562375,23166,12,14911.0
4,570467,C570867,23166,12,12607.0


## Ranking sensitivity

Compare (a) the original positive-quantity, positive-price ranking, (b) that ranking after removing each product's single largest positive line, and (c) net signed quantity using only rows with `UnitPrice > 0`. Zero-price rows are excluded from all three comparisons.

In [6]:
original_sensitivity_totals_02 = positive_population_02.groupby("StockCode")["Quantity"].sum()
ranking_a_02 = original_sensitivity_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
largest_line_by_product_02 = positive_population_02.groupby("StockCode")["Quantity"].max()
without_largest_totals_02 = (original_sensitivity_totals_02 - largest_line_by_product_02).reindex(
    original_sensitivity_totals_02.index, fill_value=0
)
ranking_b_02 = without_largest_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
positive_price_signed_rows_02 = retail_02.loc[retail_02["UnitPrice"].gt(0)]
net_signed_totals_02 = positive_price_signed_rows_02.groupby("StockCode")["Quantity"].sum()
ranking_c_02 = net_signed_totals_02.rename("quantity").reset_index().sort_values(
    ["quantity", "StockCode"], ascending=[False, True], kind="mergesort"
).reset_index(drop=True)
original_top20_codes_02 = set(ranking_a_02.head(20)["StockCode"])
for ranking_label_02, ranking_frame_02 in [
    ("(a) Original", ranking_a_02),
    ("(b) Without each product's largest line", ranking_b_02),
    ("(c) Net signed quantity; UnitPrice > 0 only", ranking_c_02),
]:
    ranking_frame_02["rank"] = range(1, len(ranking_frame_02) + 1)
    current_top20_codes_02 = set(ranking_frame_02.head(20)["StockCode"])
    two_code_ranks_02 = ranking_frame_02.loc[
        ranking_frame_02["StockCode"].astype("string").str.strip().isin(["23843", "23166"]),
        ["rank", "StockCode", "quantity"],
    ]
    print(ranking_label_02)
    display02(ranking_frame_02.head(20))
    display02(two_code_ranks_02.reset_index(drop=True))
    print(f"Original top-20 products remaining: {len(original_top20_codes_02.intersection(current_top20_codes_02))} of 20")
quantity_23843_02 = int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23843"), "quantity"].iloc[0])
rank_23843_02 = int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23843"), "rank"].iloc[0])
tied_products_23843_02 = int(ranking_b_02["quantity"].eq(quantity_23843_02).sum())
print(f"23843 quantity: {quantity_23843_02}; rank: {rank_23843_02}; products tied at that quantity: {tied_products_23843_02}")
assert quantity_23843_02 == 0
assert int(ranking_b_02.loc[ranking_b_02["StockCode"].astype("string").str.strip().eq("23166"), "quantity"].iloc[0]) == 3818
assert len(original_top20_codes_02.intersection(set(ranking_b_02.head(20)["StockCode"]))) == 18

(a) Original


,StockCode,quantity,rank
0,23843,80995,1
1,23166,78033,2
2,22197,56921,3
3,84077,55047,4
4,85099B,48474,5
5,85123A,37952,6
6,84879,36461,7
7,21212,36419,8
8,23084,30788,9
9,22492,26633,10


,rank,StockCode,quantity
0,1,23843,80995
1,2,23166,78033


Original top-20 products remaining: 20 of 20
(b) Without each product's largest line


,StockCode,quantity,rank
0,22197,52621,1
1,84077,50247,2
2,85099B,47274,3
3,85123A,36022,4
4,21212,34979,5
5,84879,33581,6
6,23084,28388,7
7,22492,25239,8
8,22616,24839,9
9,22178,23344,10


,rank,StockCode,quantity
0,319,23166,3818
1,3721,23843,0


Original top-20 products remaining: 18 of 20
(c) Net signed quantity; UnitPrice > 0 only


,StockCode,quantity,rank
0,22197,56450,1
1,84077,53847,2
2,85099B,47359,3
3,84879,36381,4
4,21212,36039,5
5,85123A,35374,6
6,23084,30680,7
7,22492,26437,8
8,22616,26111,9
9,21977,24753,10


,rank,StockCode,quantity
0,377,23166,3539
1,3800,23843,0


Original top-20 products remaining: 18 of 20
23843 quantity: 0; rank: 3721; products tied at that quantity: 128


## StockCode 84347 and file boundary

Show the recorded -9,360 return, whether any positive line for the same product predates it in this file, the file start, and the number of products whose first negative row precedes their first positive row.

In [7]:
raw_rows_84347_02 = retail_02.loc[
    retail_02["StockCode"].astype("string").str.strip().eq("84347")
].copy()
negative_rows_84347_02 = raw_rows_84347_02.loc[raw_rows_84347_02["Quantity"].lt(0)]
assert len(negative_rows_84347_02.loc[negative_rows_84347_02["Quantity"].eq(-9360)]) == 1
return_row_84347_02 = negative_rows_84347_02.loc[
    negative_rows_84347_02["Quantity"].eq(-9360)
].iloc[0]
positive_rows_84347_02 = raw_rows_84347_02.loc[
    raw_rows_84347_02["Quantity"].gt(0) & raw_rows_84347_02["UnitPrice"].gt(0)
]
return_rate_84347_02 = abs(return_row_84347_02["Quantity"]) / positive_rows_84347_02["Quantity"].sum()
earlier_sales_84347_02 = positive_rows_84347_02.loc[
    positive_rows_84347_02["InvoiceDate"].lt(return_row_84347_02["InvoiceDate"])
]
earlier_same_customer_84347_02 = earlier_sales_84347_02.loc[
    earlier_sales_84347_02["CustomerID"].eq(return_row_84347_02["CustomerID"])
]
first_negative_by_code_02 = retail_02.loc[retail_02["Quantity"].lt(0)].groupby("StockCode")["InvoiceDate"].min()
first_positive_by_code_02 = retail_02.loc[retail_02["Quantity"].gt(0)].groupby("StockCode")["InvoiceDate"].min()
first_date_comparison_02 = pandas02.concat(
    [first_negative_by_code_02.rename("first_negative"), first_positive_by_code_02.rename("first_positive")],
    axis=1,
)
negative_before_positive_count_02 = int(
    (first_date_comparison_02["first_negative"].notna()
     & first_date_comparison_02["first_positive"].notna()
     & first_date_comparison_02["first_negative"].lt(first_date_comparison_02["first_positive"])).sum()
)
print(f"File starts: {retail_02['InvoiceDate'].min()}")
print("The single -9,360 return row:")
display02(negative_rows_84347_02.loc[negative_rows_84347_02["Quantity"].eq(-9360)])
print(
    f"Positive units: {int(positive_rows_84347_02['Quantity'].sum()):,}; "
    f"return units: 9,360; return / positive units: {return_rate_84347_02:.1%}"
)
print(
    f"Earlier positive 84347 rows in the file: {len(earlier_sales_84347_02)}; "
    f"earlier rows for this customer: {len(earlier_same_customer_84347_02)}"
)
display02(
    earlier_sales_84347_02[
        ["InvoiceNo", "Quantity", "CustomerID", "InvoiceDate"]
    ].reset_index(drop=True)
)
print(f"Products whose first negative row precedes their first positive row: {negative_before_positive_count_02:,}")
assert earlier_same_customer_84347_02.empty

File starts: 2010-12-01 08:26:00
The single -9,360 return row:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
4287,C536757,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,-9360,2010-12-02 14:23:00,0.03,15838.0,United Kingdom


Positive units: 9,476; return units: 9,360; return / positive units: 98.8%
Earlier positive 84347 rows in the file: 10; earlier rows for this customer: 0


,InvoiceNo,Quantity,CustomerID,InvoiceDate
0,536408,6,14307.0,2010-12-01 11:41:00
1,536446,12,15983.0,2010-12-01 12:15:00
2,536488,1,17897.0,2010-12-01 12:31:00
3,536488,1,17897.0,2010-12-01 12:31:00
4,536488,3,17897.0,2010-12-01 12:31:00
5,536527,6,12662.0,2010-12-01 13:04:00
6,536544,4,NaN,2010-12-01 14:32:00
7,536592,1,NaN,2010-12-01 17:06:00
8,536667,6,15260.0,2010-12-02 12:09:00
9,536749,1,17976.0,2010-12-02 13:49:00


Products whose first negative row precedes their first positive row: 75


In [8]:
assert retail_02.equals(retail_snapshot_02), "The source dataframe changed during this analysis."
print("Source snapshot check passed: retail_02.equals(retail_snapshot_02).")

Source snapshot check passed: retail_02.equals(retail_snapshot_02).


## Conclusions

**Observations:** the raw rows for 23843 contain equal positive and cancellation quantities for the same customer; 23166 has a 74,215-unit largest line out of 78,033 positive-price units; ranking positions change substantially when each product's largest line is removed or signed quantities are netted.

**Interpretations and limitations:** these patterns are consistent with exceptional/cancelled transactions, but recorded quantities alone do not establish why they were entered. A product ranking is sensitive to its population definition. Returns may be unmatched or refer to sales before the file begins, and positive-quantity rows are not verified demand. Price/quantity association is not causation.

For the exact-match cleaning rules and clean-data diagnostics, see part B below.

### Part B

**Question:** Which rows remain after cleaning and exact cancellation matching?

# Clean foundation

## Filtering rules
1. Drop invoices starting with `C`.
2. Keep rows where `Quantity > 0` and `UnitPrice > 0`.
3. Keep product-like stock codes matching five digits plus optional letters.
4. Match cancellation lines to one earlier positive line with the same customer, stock code, and exact quantity; remove only matched positive lines. Each positive line can be matched once, choosing the closest earlier line.
5. Keep rows with missing `CustomerID`.

Added columns: `line_value = Quantity x UnitPrice` (recorded value, not verified revenue) and `week` (the start date of the invoice week).

Still uncertain: unmatched, partial, or altered-quantity cancellations cannot be linked by this exact rule; returns may refer to sales before the file starts; zero-price rows and nonstandard product codes are excluded by the stated rules.

In [9]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

project_root_candidates_03 = (Path.cwd(), *Path.cwd().parents)
project_root_03 = next(
    (candidate for candidate in project_root_candidates_03 if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root_03 is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root_03) not in sys.path:
    sys.path.insert(0, str(project_root_03))
from src.clean import PRODUCT_CODE, load_clean, load_raw, match_cancelled_orders

relative_data_path_03 = Path("data") / "raw" / "Online Retail.xlsx"
data_path_03 = project_root_03 / relative_data_path_03
print(f"Workbook: {relative_data_path_03}")
raw_03 = load_raw(data_path_03)
matched_positive_indices_03, matched_cancellation_indices_03 = match_cancelled_orders(raw_03)
clean_03 = load_clean(path=data_path_03, verbose=True)

assert clean_03["Quantity"].gt(0).all()
assert clean_03["UnitPrice"].gt(0).all()
assert not clean_03["InvoiceNo"].astype(str).str.startswith("C").any()
assert clean_03["StockCode"].str.match(PRODUCT_CODE).all()
print("Clean-data filter assertions passed.")

Workbook: data\raw\Online Retail.xlsx


Matched cancellations: 2,831
Rows removed by PRODUCT_CODE regex (top 20 StockCodes):


StockCode
POST            1256
DOT              710
M                572
C2               144
D                 77
S                 63
BANK CHARGES      37
AMAZONFEE         34
CRUK              16
DCGSSGIRL         13
DCGSSBOY          11
GIFT_0001_20      10
GIFT_0001_10       9
GIFT_0001_30       8
DCGS0003           5
GIFT_0001_50       4
PADS               4
GIFT_0001_40       3
B                  3
DCGS0076           2
541,909 raw rows -> 524,894 clean rows, 3,786 products, 5,351,288 units


Clean-data filter assertions passed.


## Clean-row waterfall

The table applies each clean-data filter in order, retaining only rows that pass the previous steps.

In [10]:
def apply_waterfall_step_03(frame, label, remove_mask):
    rows_before = len(frame)
    rows_removed = int(remove_mask.sum())
    remaining = frame.loc[~remove_mask].copy()
    rows_after = len(remaining)
    assert rows_removed + rows_after == rows_before
    return remaining, {
        'Step': label,
        'Rows before': rows_before,
        'Rows removed': rows_removed,
        'Rows after': rows_after,
        'Units after': int(remaining['Quantity'].sum()),
    }

waterfall_03 = raw_03.copy()
waterfall_steps_03 = []
waterfall_03, step_03 = apply_waterfall_step_03(
    waterfall_03, 'C-invoice rows',
    waterfall_03['InvoiceNo'].astype(str).str.startswith('C'),
)
waterfall_steps_03.append(step_03)
waterfall_03, step_03 = apply_waterfall_step_03(
    waterfall_03, 'Quantity or UnitPrice not positive',
    ~waterfall_03['Quantity'].gt(0) | ~waterfall_03['UnitPrice'].gt(0),
)
waterfall_steps_03.append(step_03)
waterfall_03, step_03 = apply_waterfall_step_03(
    waterfall_03, 'StockCode fails PRODUCT_CODE',
    ~waterfall_03['StockCode'].str.match(PRODUCT_CODE),
)
waterfall_steps_03.append(step_03)
waterfall_03, step_03 = apply_waterfall_step_03(
    waterfall_03, 'Removed by cancellation matcher',
    waterfall_03.index.isin(matched_positive_indices_03),
)
waterfall_steps_03.append(step_03)
waterfall_table_03 = pd.DataFrame(waterfall_steps_03)
display(waterfall_table_03)
assert len(waterfall_03) == len(clean_03)
assert int(waterfall_03['Quantity'].sum()) == int(clean_03['Quantity'].sum())

,Step,Rows before,Rows removed,Rows after,Units after
0,C-invoice rows,541909,9288,532621,5454024
1,Quantity or UnitPrice not positive,532621,2517,530104,5588376
2,StockCode fails PRODUCT_CODE,530104,2379,527725,5576996
3,Removed by cancellation matcher,527725,2831,524894,5351288


## Exact cancellation matching summary

A cancellation is considered only when its invoice starts with `C`, quantity is negative, unit price is positive, and customer ID is present. Matching also requires an earlier positive line with the same customer, stock code, and absolute quantity.

In [11]:
considered_cancellations_03 = (
    raw_03["InvoiceNo"].astype(str).str.startswith("C")
    & raw_03["Quantity"].lt(0)
    & raw_03["UnitPrice"].gt(0)
    & raw_03["CustomerID"].notna()
)
matched_pairs_03 = pd.DataFrame({
    "InvoiceNo": raw_03.loc[matched_positive_indices_03, "InvoiceNo"].to_numpy(),
    "C-InvoiceNo": raw_03.loc[matched_cancellation_indices_03, "InvoiceNo"].to_numpy(),
    "StockCode": raw_03.loc[matched_positive_indices_03, "StockCode"].to_numpy(),
    "Quantity": raw_03.loc[matched_positive_indices_03, "Quantity"].to_numpy(),
    "CustomerID": raw_03.loc[matched_positive_indices_03, "CustomerID"].to_numpy(),
})
matched_quantities_03 = matched_pairs_03["Quantity"]
print(f"Cancellation lines considered: {int(considered_cancellations_03.sum()):,}")
print(f"Cancellation lines matched: {len(matched_cancellation_indices_03):,}")
print(f"Cancellation lines unmatched: {int(considered_cancellations_03.sum()) - len(matched_cancellation_indices_03):,}")
print(f"Positive lines excluded: {len(matched_positive_indices_03):,}")
print(f"Units in positive lines excluded: {int(raw_03.loc[matched_positive_indices_03, 'Quantity'].sum()):,}")
print("10 largest matched pairs:")
display(
    matched_pairs_03.sort_values(
        ["Quantity", "InvoiceNo"], ascending=[False, True], kind="mergesort"
    ).head(10).reset_index(drop=True)
)
print(
    "Matched quantity distribution: "
    f"min={matched_quantities_03.min():,}, "
    f"median={matched_quantities_03.median():g}, "
    f"90th percentile={matched_quantities_03.quantile(0.9):g}, "
    f"max={matched_quantities_03.max():,}, "
    f"pairs with quantity <= 5={int(matched_quantities_03.le(5).sum()):,}"
)

Cancellation lines considered: 8,905
Cancellation lines matched: 2,831
Cancellation lines unmatched: 6,074
Positive lines excluded: 2,831
Units in positive lines excluded: 225,708
10 largest matched pairs:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,581483,C581484,23843,80995,16446.0
1,541431,C541433,23166,74215,12346.0
2,540815,C550456,21108,3114,15749.0
3,540815,C550456,21175,2000,15749.0
4,540815,C550456,85123A,1930,15749.0
5,556484,C556522,22920,1515,16938.0
6,570094,C570556,20971,1296,16029.0
7,570097,C570556,20971,1296,16029.0
8,569214,C569552,15034,1200,14533.0
9,570097,C570556,22147,840,16029.0


Matched quantity distribution: min=1, median=6, 90th percentile=40, max=80,995, pairs with quantity <= 5=1,300


## Matched pairs: time gap, size, full-invoice cancellation

In [ ]:
matched_pairs_03b = pd.DataFrame({
    "positive_index": matched_positive_indices_03,
    "cancellation_index": matched_cancellation_indices_03,
})
matched_pairs_03b["InvoiceNo"] = raw_03.loc[matched_pairs_03b["positive_index"], "InvoiceNo"].to_numpy()
matched_pairs_03b["C-InvoiceNo"] = raw_03.loc[matched_pairs_03b["cancellation_index"], "InvoiceNo"].to_numpy()
matched_pairs_03b["StockCode"] = raw_03.loc[matched_pairs_03b["positive_index"], "StockCode"].to_numpy()
matched_pairs_03b["Quantity"] = raw_03.loc[matched_pairs_03b["positive_index"], "Quantity"].to_numpy()
matched_pairs_03b["CustomerID"] = raw_03.loc[matched_pairs_03b["positive_index"], "CustomerID"].to_numpy()
matched_pairs_03b["positive_date"] = raw_03.loc[matched_pairs_03b["positive_index"], "InvoiceDate"].to_numpy()
matched_pairs_03b["cancellation_date"] = raw_03.loc[matched_pairs_03b["cancellation_index"], "InvoiceDate"].to_numpy()
matched_pairs_03b["gap"] = matched_pairs_03b["cancellation_date"] - matched_pairs_03b["positive_date"]
matched_pairs_03b["gap_days"] = matched_pairs_03b["gap"].dt.total_seconds() / 86400

print("Gap percentiles (days):")
display(matched_pairs_03b["gap_days"].quantile([0, 0.25, 0.5, 0.75, 0.9, 1]).rename(index={0.0: "min", 0.25: "25th percentile", 0.5: "median", 0.75: "75th percentile", 0.9: "90th percentile", 1.0: "max"}).to_frame("days"))

matched_pairs_03b["gap_bucket"] = pd.Series(pd.NA, index=matched_pairs_03b.index, dtype="string")
matched_pairs_03b.loc[matched_pairs_03b["gap"].lt(pd.Timedelta(hours=1)), "gap_bucket"] = "<1 hour"
matched_pairs_03b.loc[matched_pairs_03b["gap"].ge(pd.Timedelta(hours=1)) & matched_pairs_03b["gap"].lt(pd.Timedelta(days=1)), "gap_bucket"] = "same day (>=1 hour)"
matched_pairs_03b.loc[matched_pairs_03b["gap"].ge(pd.Timedelta(days=1)) & matched_pairs_03b["gap"].le(pd.Timedelta(days=7)), "gap_bucket"] = "1-7 days"
matched_pairs_03b.loc[matched_pairs_03b["gap"].gt(pd.Timedelta(days=7)) & matched_pairs_03b["gap"].le(pd.Timedelta(days=30)), "gap_bucket"] = "8-30 days"
matched_pairs_03b.loc[matched_pairs_03b["gap"].gt(pd.Timedelta(days=30)), "gap_bucket"] = ">30 days"
matched_pairs_03b["quantity_bucket"] = pd.Series(pd.NA, index=matched_pairs_03b.index, dtype="string")
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].le(5), "quantity_bucket"] = "<=5"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].between(6, 50), "quantity_bucket"] = "6-50"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].between(51, 999), "quantity_bucket"] = "51-999"
matched_pairs_03b.loc[matched_pairs_03b["Quantity"].ge(1000), "quantity_bucket"] = ">=1000"
gap_bucket_order_03b = ["<1 hour", "same day (>=1 hour)", "1-7 days", "8-30 days", ">30 days"]
quantity_bucket_order_03b = ["<=5", "6-50", "51-999", ">=1000"]

print("Pair counts and units by gap bucket:")
display(matched_pairs_03b.groupby("gap_bucket", observed=False).agg(pairs=("Quantity", "size"), units=("Quantity", "sum")).reindex(gap_bucket_order_03b, fill_value=0))
print("Pair counts and units by quantity bucket:")
display(matched_pairs_03b.groupby("quantity_bucket", observed=False).agg(pairs=("Quantity", "size"), units=("Quantity", "sum")).reindex(quantity_bucket_order_03b, fill_value=0))

print("Pair-count crosstab: gap bucket x quantity bucket")
matched_pair_counts_crosstab_03b = matched_pairs_03b.groupby(["gap_bucket", "quantity_bucket"], observed=False).size().unstack(fill_value=0).reindex(index=gap_bucket_order_03b, columns=quantity_bucket_order_03b, fill_value=0)
display(matched_pair_counts_crosstab_03b)
print("Unit-sum crosstab: gap bucket x quantity bucket")
matched_pair_units_crosstab_03b = matched_pairs_03b.groupby(["gap_bucket", "quantity_bucket"], observed=False)["Quantity"].sum().unstack(fill_value=0).reindex(index=gap_bucket_order_03b, columns=quantity_bucket_order_03b, fill_value=0)
display(matched_pair_units_crosstab_03b)

matched_pairs_selected_03b = matched_pairs_03b.loc[matched_pairs_03b["Quantity"].ge(1000) | matched_pairs_03b["gap"].lt(pd.Timedelta(hours=1))].copy()
matched_pairs_selected_03b["gap_hours"] = matched_pairs_selected_03b["gap"].dt.total_seconds() / 3600
matched_pairs_selected_03b = matched_pairs_selected_03b.sort_values(["Quantity", "gap"], ascending=[False, True], kind="mergesort")
print(f"Pairs meeting Quantity >= 1000 or gap < 1 hour: {len(matched_pairs_selected_03b)}; shown: {min(40, len(matched_pairs_selected_03b))}")
display(matched_pairs_selected_03b[["InvoiceNo", "C-InvoiceNo", "StockCode", "Quantity", "CustomerID", "positive_date", "cancellation_date", "gap", "gap_hours"]].head(40).reset_index(drop=True))

matched_positive_index_set_03b = set(matched_positive_indices_03)
matched_invoice_rows_03b = []
for matched_invoice_03b in matched_pairs_03b["InvoiceNo"].drop_duplicates():
    invoice_raw_rows_03b = raw_03.loc[raw_03["InvoiceNo"].eq(matched_invoice_03b)]
    invoice_matched_rows_03b = invoice_raw_rows_03b.loc[invoice_raw_rows_03b.index.isin(matched_positive_index_set_03b)]
    matched_invoice_rows_03b.append({
        "InvoiceNo": matched_invoice_03b,
        "raw_lines": len(invoice_raw_rows_03b),
        "matched_lines": len(invoice_matched_rows_03b),
        "matched_units": int(invoice_matched_rows_03b["Quantity"].sum()),
        "fully_cancelled": len(invoice_raw_rows_03b) == len(invoice_matched_rows_03b),
    })
matched_invoice_detail_03b = pd.DataFrame(matched_invoice_rows_03b)
matched_invoice_detail_03b["invoice_status"] = matched_invoice_detail_03b["fully_cancelled"].map({True: "fully cancelled", False: "partly cancelled"})
print("Matched positive invoices: full vs partial cancellation:")
display(matched_invoice_detail_03b.groupby("invoice_status", sort=False).agg(invoices=("InvoiceNo", "size"), raw_lines=("raw_lines", "sum"), matched_lines=("matched_lines", "sum"), units=("matched_units", "sum")).reindex(["fully cancelled", "partly cancelled"], fill_value=0))

print("Five matched pairs for StockCode 23166 with gaps:")
display(matched_pairs_03b.loc[matched_pairs_03b["StockCode"].eq("23166"), ["InvoiceNo", "C-InvoiceNo", "StockCode", "Quantity", "CustomerID", "positive_date", "cancellation_date", "gap", "gap_days"]].sort_values(["positive_date", "InvoiceNo"], kind="mergesort").reset_index(drop=True))

Gap percentiles (days):


,days
min,0.000694
25th percentile,1.997917
median,9.199306
75th percentile,28.902778
90th percentile,70.095833
max,368.187500


Pair counts and units by gap bucket:


,pairs,units
gap_bucket,,
<1 hour,433,172593
same day (>=1 hour),208,7128
1-7 days,603,15413
8-30 days,895,11516
>30 days,692,19058


Pair counts and units by quantity bucket:


,pairs,units
quantity_bucket,,
<=5,1300,2478
6-50,1321,21233
51-999,201,34436
>=1000,9,167561


... (6 outputs left out)

Matched positive invoices: full vs partial cancellation:


,invoices,raw_lines,matched_lines,units
invoice_status,,,,
fully cancelled,120,938,938,197851
partly cancelled,1197,41242,1893,27857


Five matched pairs for StockCode 23166 with gaps:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID,positive_date,cancellation_date,gap,gap_days
0,541431,C541433,23166,74215,12346.0,2011-01-18 10:01:00,2011-01-18 10:17:00,0 days 00:16:00,0.011111
1,553607,C557508,23166,240,16684.0,2011-05-18 10:47:00,2011-06-20 16:13:00,33 days 05:26:00,33.226389
2,560491,C560540,23166,1,12415.0,2011-07-19 10:51:00,2011-07-19 12:26:00,0 days 01:35:00,0.065972
3,562374,C562375,23166,12,14911.0,2011-08-04 14:40:00,2011-08-04 14:46:00,0 days 00:06:00,0.004167
4,570467,C570867,23166,12,12607.0,2011-10-10 16:06:00,2011-10-12 16:17:00,2 days 00:11:00,2.007639


## Prices in matched pairs

This compares the positive line price with its matched cancellation-line price.

In [13]:
matched_pair_prices_03 = pd.DataFrame({
    'Positive UnitPrice': raw_03.loc[matched_positive_indices_03, 'UnitPrice'].to_numpy(),
    'Cancellation UnitPrice': raw_03.loc[matched_cancellation_indices_03, 'UnitPrice'].to_numpy(),
})
matched_pair_prices_03['Price ratio (cancellation / positive)'] = (
    matched_pair_prices_03['Cancellation UnitPrice']
    / matched_pair_prices_03['Positive UnitPrice']
)
equal_price_count_03 = int(
    matched_pair_prices_03['Positive UnitPrice'].eq(
        matched_pair_prices_03['Cancellation UnitPrice']
    ).sum()
)
price_comparison_03 = pd.DataFrame([{
    'Matched pairs': len(matched_pair_prices_03),
    'Equal prices': equal_price_count_03,
    'Different prices': len(matched_pair_prices_03) - equal_price_count_03,
}])
display(price_comparison_03)
price_ratio_distribution_03 = (
    matched_pair_prices_03['Price ratio (cancellation / positive)']
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9])
    .to_frame('Price ratio')
)
display(price_ratio_distribution_03)

,Matched pairs,Equal prices,Different prices
0,2831,2735,96


,Price ratio
count,2831.000000
mean,0.995315
std,0.077000
min,0.007621
25%,1.000000
50%,1.000000
75%,1.000000
90%,1.000000
max,3.593333


## Cancellation matching summary

Considered cancellations are split by product-code status; matched share is the matched count divided by the group count.

In [14]:
product_code_mask_03 = raw_03['StockCode'].str.match(PRODUCT_CODE)
matched_cancellation_mask_03 = raw_03.index.isin(matched_cancellation_indices_03)
cancellation_summary_rows_03 = []
for group_name_03, group_mask_03 in [
    ('PRODUCT_CODE match', product_code_mask_03),
    ('PRODUCT_CODE non-match', ~product_code_mask_03),
]:
    group_indices_03 = raw_03.index[
        considered_cancellations_03 & group_mask_03
    ]
    group_matched_03 = int(group_indices_03.isin(matched_cancellation_indices_03).sum())
    group_count_03 = len(group_indices_03)
    cancellation_summary_rows_03.append({
        'StockCode group': group_name_03,
        'Count': group_count_03,
        'Matched': group_matched_03,
        'Unmatched': group_count_03 - group_matched_03,
        'Matched share': group_matched_03 / group_count_03 if group_count_03 else 0,
    })
cancellation_summary_03 = pd.DataFrame(cancellation_summary_rows_03)
display(cancellation_summary_03)

missing_customer_cancellations_03 = (
    raw_03['InvoiceNo'].astype(str).str.startswith('C')
    & raw_03['Quantity'].lt(0)
    & raw_03['UnitPrice'].gt(0)
    & raw_03['CustomerID'].isna()
)
excluded_cancellations_03 = pd.DataFrame([{
    'Excluded C-invoice rows (missing CustomerID)': int(missing_customer_cancellations_03.sum())
}])
display(excluded_cancellations_03)
assert int(considered_cancellations_03.sum()) == 8905
assert int(missing_customer_cancellations_03.sum()) == 383
assert int(considered_cancellations_03.sum() + missing_customer_cancellations_03.sum()) == 9288

,StockCode group,Count,Matched,Unmatched,Matched share
0,PRODUCT_CODE match,8539,2831,5708,0.331538
1,PRODUCT_CODE non-match,366,0,366,0.000000


,Excluded C-invoice rows (missing CustomerID)
0,383


## Clean population and selected code checks

The table separates raw row counts from rows eligible under the unchanged positive-quantity, positive-price, and product-code rules, before exact-match exclusions.

In [15]:
eligible_before_matching_03 = raw_03.loc[
    ~raw_03["InvoiceNo"].astype(str).str.startswith("C")
    & raw_03["Quantity"].gt(0)
    & raw_03["UnitPrice"].gt(0)
    & raw_03["StockCode"].str.match(PRODUCT_CODE)
]
eligible_matched_03 = eligible_before_matching_03.loc[
    eligible_before_matching_03.index.isin(matched_positive_indices_03)
]
selected_codes_03 = ["23843", "23166", "84347"]
code_rows_03 = []
for stock_code_03 in selected_codes_03:
    raw_code_rows_03 = raw_03.loc[raw_03["StockCode"].eq(stock_code_03)]
    eligible_code_rows_03 = eligible_before_matching_03.loc[
        eligible_before_matching_03["StockCode"].eq(stock_code_03)
    ]
    matched_code_rows_03 = raw_03.loc[
        raw_03.index.isin(matched_positive_indices_03)
        & raw_03["StockCode"].eq(stock_code_03)
    ]
    eligible_matched_code_rows_03 = eligible_matched_03.loc[
        eligible_matched_03["StockCode"].eq(stock_code_03)
    ]
    clean_code_rows_03 = clean_03.loc[clean_03["StockCode"].eq(stock_code_03)]
    code_rows_03.append({
        "StockCode": stock_code_03,
        "Raw rows": len(raw_code_rows_03),
        "Eligible before matching": len(eligible_code_rows_03),
        "Matched positive lines": len(matched_code_rows_03),
        "Matched units": int(matched_code_rows_03["Quantity"].sum()),
        "Eligible matched lines": len(eligible_matched_code_rows_03),
        "Eligible matched units": int(eligible_matched_code_rows_03["Quantity"].sum()),
        "Clean rows": len(clean_code_rows_03),
        "Clean units": int(clean_code_rows_03["Quantity"].sum()),
    })
code_summary_03 = pd.DataFrame(code_rows_03)
display(code_summary_03)

code_23843_match_03 = raw_03.loc[raw_03["StockCode"].eq("23843") & raw_03.index.isin(matched_positive_indices_03)]
code_23166_match_03 = eligible_matched_03.loc[eligible_matched_03["StockCode"].eq("23166")]
code_84347_eligible_03 = eligible_before_matching_03.loc[eligible_before_matching_03["StockCode"].eq("84347")]
code_84347_clean_03 = clean_03.loc[clean_03["StockCode"].eq("84347")]
assert clean_03["StockCode"].eq("23843").sum() == 0
assert int(code_23166_match_03["Quantity"].eq(74215).sum()) == 1
print("Exact eligible matched pairs for 23166:")
display(matched_pairs_03.loc[matched_pairs_03["StockCode"].eq("23166")].reset_index(drop=True))
assert len(code_84347_eligible_03) == len(code_84347_clean_03)
assert code_84347_eligible_03["Quantity"].sum() == code_84347_clean_03["Quantity"].sum()
print(
    "Confirmed: 23843 has 0 clean rows; 84347's eligible rows are fully kept. "
    f"For 23166, {len(code_23166_match_03)} eligible lines match exactly, "
    "including the 74,215-unit line."
)

old_clean_rows_03, old_clean_products_03, old_clean_units_03 = 527003, 3897, 5408492
new_clean_rows_03 = len(clean_03)
new_clean_products_03 = clean_03["StockCode"].nunique()
new_clean_units_03 = int(clean_03["Quantity"].sum())
print(f"Clean rows: {new_clean_rows_03:,} (difference from old: {new_clean_rows_03 - old_clean_rows_03:+,})")
print(f"Products: {new_clean_products_03:,} (difference from old: {new_clean_products_03 - old_clean_products_03:+,})")
print(f"Total units: {new_clean_units_03:,} (difference from old: {new_clean_units_03 - old_clean_units_03:+,})")

,StockCode,Raw rows,Eligible before matching,Matched positive lines,Matched units,Eligible matched lines,Eligible matched units,Clean rows,Clean units
0,23843,2,1,1,80995,1,80995,0,0
1,23166,260,250,5,74480,5,74480,245,3553
2,84347,475,471,0,0,0,0,471,9476


Exact eligible matched pairs for 23166:


,InvoiceNo,C-InvoiceNo,StockCode,Quantity,CustomerID
0,541431,C541433,23166,74215,12346.0
1,553607,C557508,23166,240,16684.0
2,560491,C560540,23166,1,12415.0
3,562374,C562375,23166,12,14911.0
4,570467,C570867,23166,12,12607.0


Confirmed: 23843 has 0 clean rows; 84347's eligible rows are fully kept. For 23166, 5 eligible lines match exactly, including the 74,215-unit line.
Clean rows: 524,894 (difference from old: -2,109)
Products: 3,786 (difference from old: -111)
Total units: 5,351,288 (difference from old: -57,204)


### Part C

**Question:** What unmatched cancellations have a larger eligible earlier purchase?

# Partial-return candidates among unmatched cancellations

This screen examines cancellations that the exact-quantity matcher did not match. It looks for eligible positive lines for the same customer and stock code dated strictly earlier. A larger earlier line is a possible partial-return candidate, not proof that the cancellation refers to that line. Recorded quantity is not verified demand.

The matcher selects cancellation lines with a `C` invoice, negative quantity, positive unit price, and a present customer ID. Positive comparison lines additionally require a non-`C` invoice, positive quantity and unit price, a present customer ID and invoice date, and a stock code matching the project's product-code regex.

Each unmatched cancellation belongs to exactly one of five groups: a larger earlier eligible purchase exists; no larger purchase exists but an equal-quantity earlier purchase does; only strictly smaller earlier purchases exist; no earlier purchase exists and the cancellation stock code matches the product-code regex; or no earlier purchase exists and the cancellation stock code fails the regex. Earlier lines must share customer and stock code and have a strictly earlier invoice date. Positive lines are candidate evidence, not proof of linked returns; recorded quantity is not verified demand.

In [16]:
from bisect import bisect_left
from hashlib import sha256
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
project_root = next(
    (candidate for candidate in project_root_candidates if (candidate / "src" / "clean.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root from the current directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.clean import PRODUCT_CODE, load_raw, match_cancelled_orders

workbook_path = project_root / "data" / "raw" / "Online Retail.xlsx"
workbook_hash_before = sha256(workbook_path.read_bytes()).hexdigest()
raw = load_raw(workbook_path)
raw_snapshot = raw.copy(deep=True)
matched_positive_index, matched_cancellation_index = match_cancelled_orders(raw)
matched_positive_index_set = set(matched_positive_index)

print(f"Workbook: {workbook_path.relative_to(project_root)}")
print(f"Raw rows: {len(raw):,}")
print(f"Matched cancellation lines: {len(matched_cancellation_index):,}")

Workbook: data\raw\Online Retail.xlsx
Raw rows: 541,909
Matched cancellation lines: 2,831


In [17]:
cancellation_mask = (
    raw["InvoiceNo"].astype(str).str.startswith("C")
    & raw["Quantity"].lt(0)
    & raw["UnitPrice"].gt(0)
    & raw["CustomerID"].notna()
)
cancellation_lines = raw.loc[cancellation_mask].copy()
cancellation_lines["product_code_matches"] = cancellation_lines["StockCode"].str.match(PRODUCT_CODE)
unmatched_cancellations = cancellation_lines.loc[
    ~cancellation_lines.index.isin(matched_cancellation_index)
].copy()

positive_mask = (
    ~raw["InvoiceNo"].astype(str).str.startswith("C")
    & raw["Quantity"].gt(0)
    & raw["CustomerID"].notna()
    & raw["InvoiceDate"].notna()
    & raw["UnitPrice"].gt(0)
    & raw["StockCode"].str.match(PRODUCT_CODE)
)
eligible_positive_lines = raw.loc[positive_mask]

positive_by_customer_product = {}
for key, lines in eligible_positive_lines.groupby(["CustomerID", "StockCode"], sort=False):
    ordered = lines.sort_values("InvoiceDate", kind="mergesort")
    positive_by_customer_product[key] = (
        ordered["InvoiceDate"].tolist(),
        ordered["Quantity"].tolist(),
        ordered.index.tolist(),
    )

group_larger = "larger earlier purchase exists"
group_equal = "no larger earlier purchase; equal-quantity earlier purchase exists"
group_smaller = "only strictly smaller earlier purchases exist"
group_none_valid_code = "no earlier purchase; product-code StockCode"
group_none_invalid_code = "no earlier purchase; StockCode fails PRODUCT_CODE"
group_order = [
    group_larger,
    group_equal,
    group_smaller,
    group_none_valid_code,
    group_none_invalid_code,
]
classified_rows = []

for cancellation in unmatched_cancellations.itertuples():
    returned_units = -cancellation.Quantity
    key = (cancellation.CustomerID, cancellation.StockCode)
    candidate_dates, candidate_quantities, candidate_indexes = positive_by_customer_product.get(
        key, ([], [], [])
    )
    earlier_stop = (
        bisect_left(candidate_dates, cancellation.InvoiceDate)
        if pd.notna(cancellation.InvoiceDate)
        else 0
    )
    earlier_quantities = candidate_quantities[:earlier_stop]
    earlier_indexes = candidate_indexes[:earlier_stop]
    equal_indexes = [
        index
        for index, quantity in zip(earlier_indexes, earlier_quantities)
        if quantity == returned_units
    ]
    matched_equal_count = sum(index in matched_positive_index_set for index in equal_indexes)

    if any(quantity > returned_units for quantity in earlier_quantities):
        group = group_larger
    elif equal_indexes:
        group = group_equal
    elif earlier_quantities:
        group = group_smaller
    elif cancellation.product_code_matches:
        group = group_none_valid_code
    else:
        group = group_none_invalid_code

    classified_rows.append(
        {
            "cancellation_index": cancellation.Index,
            "InvoiceNo": cancellation.InvoiceNo,
            "InvoiceDate": cancellation.InvoiceDate,
            "CustomerID": cancellation.CustomerID,
            "StockCode": cancellation.StockCode,
            "returned_units": returned_units,
            "earlier_purchase_count": len(earlier_quantities),
            "largest_earlier_quantity": max(earlier_quantities, default=None),
            "equal_sized_earlier_count": len(equal_indexes),
            "matched_equal_earlier_count": matched_equal_count,
            "group": group,
        }
    )

classified = pd.DataFrame(classified_rows)
if classified.empty:
    classified = pd.DataFrame(
        columns=[
            "cancellation_index", "InvoiceNo", "InvoiceDate", "CustomerID", "StockCode",
            "returned_units", "earlier_purchase_count", "largest_earlier_quantity",
            "equal_sized_earlier_count", "matched_equal_earlier_count", "group",
        ]
    )

print(f"Cancellation lines considered by matcher: {len(cancellation_lines):,}")
print(f"Unmatched cancellation lines screened: {len(unmatched_cancellations):,}")
print(f"Eligible positive lines: {len(eligible_positive_lines):,}")

Cancellation lines considered by matcher: 8,905
Unmatched cancellation lines screened: 6,074
Eligible positive lines: 396,337


In [18]:
summary_rows = []
for group in group_order:
    rows = classified.loc[classified["group"].eq(group)]
    summary_rows.append(
        {
            "group": group,
            "cancellation_count": len(rows),
            "returned_units": int(rows["returned_units"].sum()),
            "rows_with_equal_earlier_lines": int(rows["equal_sized_earlier_count"].gt(0).sum()),
            "all_equal_lines_matched": int(
                (
                    rows["equal_sized_earlier_count"].gt(0)
                    & rows["matched_equal_earlier_count"].eq(rows["equal_sized_earlier_count"])
                ).sum()
            ),
            "at_least_one_equal_line_unmatched": int(
                (
                    rows["equal_sized_earlier_count"].gt(rows["matched_equal_earlier_count"])
                ).sum()
            ),
        }
    )
summary = pd.DataFrame(summary_rows)
display(summary)

example_columns = [
    "InvoiceNo", "InvoiceDate", "CustomerID", "StockCode", "returned_units",
    "earlier_purchase_count", "largest_earlier_quantity", "equal_sized_earlier_count",
    "matched_equal_earlier_count",
]
for group in group_order:
    print(f"\nExamples: {group}")
    examples = classified.loc[classified["group"].eq(group), example_columns].head(5)
    display(examples.reset_index(drop=True))

def report_assertion(label, check):
    try:
        check()
    except AssertionError as error:
        print(f"FAIL: {label}: {error}")
    else:
        print(f"PASS: {label}")

group_equal_rows = classified.loc[classified["group"].eq(group_equal)].copy()
group_equal_rows["equal_lines_status"] = "at least one equal line unmatched"
group_equal_rows.loc[
    group_equal_rows["matched_equal_earlier_count"].eq(
        group_equal_rows["equal_sized_earlier_count"]
    ),
    "equal_lines_status",
] = "all equal lines already matched"
print("\nEqual-quantity group (b): matched-status counts")
display(
    group_equal_rows["equal_lines_status"]
    .value_counts()
    .reindex(["all equal lines already matched", "at least one equal line unmatched"], fill_value=0)
    .rename_axis("equal_lines_status")
    .to_frame("cancellation_count")
)
for status in ["all equal lines already matched", "at least one equal line unmatched"]:
    print(f"\nExamples: group (b), {status}")
    status_examples = group_equal_rows.loc[
        group_equal_rows["equal_lines_status"].eq(status), example_columns
    ].head(5)
    print(f"Available rows: {len(group_equal_rows.loc[group_equal_rows['equal_lines_status'].eq(status)]):,}; showing {len(status_examples):,} of 5 requested.")
    if status_examples.empty:
        print("No examples exist for this status in the screened data.")
    display(status_examples.reset_index(drop=True))

group_larger_equal_rows = classified.loc[
    classified["group"].eq(group_larger) & classified["equal_sized_earlier_count"].gt(0)
].copy()
group_larger_equal_rows["equal_lines_status"] = "at least one equal line unmatched"
group_larger_equal_rows.loc[
    group_larger_equal_rows["matched_equal_earlier_count"].eq(
        group_larger_equal_rows["equal_sized_earlier_count"]
    ),
    "equal_lines_status",
] = "all equal lines already matched"
print("\nGroup (a): cancellations that also have equal-quantity earlier lines")
print(f"Rows with an equal earlier line: {len(group_larger_equal_rows):,}")
display(group_larger_equal_rows["equal_lines_status"].value_counts().rename_axis("equal_lines_status").to_frame("cancellation_count"))
print("Examples: group (a) rows with equal-quantity earlier lines")
display(group_larger_equal_rows[example_columns].head(5).reset_index(drop=True))

def assert_groups_sum_to_expected():
    actual = int(summary["cancellation_count"].sum())
    assert actual == 6_074, f"five group counts sum to {actual:,}, expected 6,074"

def assert_every_unmatched_cancellation_classified_once():
    actual_indexes = classified["cancellation_index"]
    assert len(actual_indexes) == len(unmatched_cancellations), (
        f"classified {len(actual_indexes):,} rows for {len(unmatched_cancellations):,} unmatched cancellations"
    )
    assert actual_indexes.is_unique, "a cancellation was classified more than once"
    assert set(actual_indexes) == set(unmatched_cancellations.index), (
        "the classified indexes differ from unmatched cancellation indexes"
    )
    assert classified["group"].isin(group_order).all(), "an unmatched cancellation has no valid group"

def assert_raw_dataframe_unchanged():
    pd.testing.assert_frame_equal(raw, raw_snapshot)

def assert_workbook_unchanged():
    current_hash = sha256(workbook_path.read_bytes()).hexdigest()
    assert current_hash == workbook_hash_before, "workbook SHA-256 changed during this analysis"

print("\nValidation")
report_assertion("five group counts sum to 6,074", assert_groups_sum_to_expected)
report_assertion(
    "every unmatched cancellation is classified exactly once",
    assert_every_unmatched_cancellation_classified_once,
)
report_assertion("loaded raw DataFrame unchanged", assert_raw_dataframe_unchanged)
report_assertion("raw workbook unchanged", assert_workbook_unchanged)

,group,cancellation_count,returned_units,rows_with_equal_earlier_lines,all_equal_lines_matched,at_least_one_equal_line_unmatched
0,larger earlier purchase exists,4572,23914,16,16,0
1,no larger earlier purchase; equal-quantity ear...,21,131,21,21,0
2,only strictly smaller earlier purchases exist,84,1678,0,0,0
3,no earlier purchase; product-code StockCode,1031,18053,0,0,0
4,no earlier purchase; StockCode fails PRODUCT_CODE,366,5324,0,0,0



Examples: larger earlier purchase exists


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C536506,2010-12-01 12:38:00,17897.0,22960,6,1,8.0,0,0
1,C536737,2010-12-02 13:05:00,15922.0,22798,2,1,8.0,0,0
2,C536758,2010-12-02 14:25:00,13767.0,21314,1,1,8.0,0,0
3,C536826,2010-12-02 17:30:00,17924.0,35004B,2,1,12.0,0,0
4,C536826,2010-12-02 17:30:00,17924.0,35004B,3,1,12.0,0,0



Examples: no larger earlier purchase; equal-quantity earlier purchase exists


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C537406,2010-12-06 14:53:00,15502.0,22926,4,1,4.0,1,1
1,C537406,2010-12-06 14:53:00,15502.0,22927,4,1,4.0,1,1
2,C537406,2010-12-06 14:53:00,15502.0,22847,4,1,4.0,1,1
3,C537406,2010-12-06 14:53:00,15502.0,22849,4,1,4.0,1,1
4,C537408,2010-12-06 14:55:00,15502.0,47580,6,1,6.0,1,1



Examples: only strictly smaller earlier purchases exist


... (13 outputs left out)

Examples: group (a) rows with equal-quantity earlier lines


,InvoiceNo,InvoiceDate,CustomerID,StockCode,returned_units,earlier_purchase_count,largest_earlier_quantity,equal_sized_earlier_count,matched_equal_earlier_count
0,C548355,2011-03-30 15:06:00,12921.0,22423,1,2,4.0,1,1
1,C549019,2011-04-05 15:22:00,12474.0,22236,1,2,2.0,1,1
2,C551659,2011-05-03 12:14:00,13668.0,85053,1,3,48.0,1,1
3,C553687,2011-05-18 14:01:00,15311.0,22646,1,3,72.0,1,1
4,C556643,2011-06-13 15:48:00,17841.0,23146,2,5,3.0,1,1



Validation
PASS: five group counts sum to 6,074
PASS: every unmatched cancellation is classified exactly once


PASS: loaded raw DataFrame unchanged
PASS: raw workbook unchanged


In [19]:
months = pd.period_range("2010-12", "2011-12", freq="M")

group_monthly = classified.loc[classified["group"].eq(group_none_valid_code)].copy()
group_monthly["calendar_month"] = group_monthly["InvoiceDate"].dt.to_period("M")
all_unmatched_monthly = unmatched_cancellations.copy()
all_unmatched_monthly["calendar_month"] = all_unmatched_monthly["InvoiceDate"].dt.to_period("M")

group_monthly_counts = group_monthly.groupby("calendar_month").size().reindex(months, fill_value=0)
group_monthly_units = (
    group_monthly.groupby("calendar_month")["returned_units"]
    .sum()
    .reindex(months, fill_value=0)
    .astype("int64")
)
all_unmatched_monthly_counts = (
    all_unmatched_monthly.groupby("calendar_month")
    .size()
    .reindex(months, fill_value=0)
)
monthly_table = pd.DataFrame(
    {
        "calendar_month": months.astype(str),
        "group_cancellations": group_monthly_counts.to_numpy(),
        "returned_units": group_monthly_units.to_numpy(),
        "all_unmatched_cancellations": all_unmatched_monthly_counts.to_numpy(),
        "share_of_group_total": (
            group_monthly_counts / group_monthly_counts.sum()
        ).to_numpy(),
    }
)
display(monthly_table.style.format({"share_of_group_total": "{:.1%}"}))

def assert_monthly_group_count_sum():
    actual = int(monthly_table["group_cancellations"].sum())
    assert actual == 1_031, f"monthly group counts sum to {actual:,}, expected 1,031"

def assert_monthly_unmatched_count_sum():
    actual = int(monthly_table["all_unmatched_cancellations"].sum())
    assert actual == 6_074, f"monthly unmatched counts sum to {actual:,}, expected 6,074"

print("\nMonthly validation")
report_assertion("monthly group counts sum to 1,031", assert_monthly_group_count_sum)
report_assertion("monthly all-unmatched counts sum to 6,074", assert_monthly_unmatched_count_sum)

,calendar_month,group_cancellations,returned_units,all_unmatched_cancellations,share_of_group_total
0,2010-12,371,12586,556,36.0%
1,2011-01,173,1469,512,16.8%
2,2011-02,95,347,349,9.2%
3,2011-03,65,151,438,6.3%
4,2011-04,38,1476,359,3.7%
5,2011-05,22,29,412,2.1%
6,2011-06,25,42,435,2.4%
7,2011-07,53,835,408,5.1%
8,2011-08,19,75,469,1.8%
9,2011-09,49,371,600,4.8%



Monthly validation
PASS: monthly group counts sum to 1,031
PASS: monthly all-unmatched counts sum to 6,074


The table reports monthly counts and returned units for cancellations in the “no earlier purchase; product-code StockCode” group, all unmatched cancellations, and the group’s share of its total from 2010-12 through 2011-12.

## Interpreting the cleaned ranking

The raw ranking put `23843` at 80,995 units and `23166` at 78,033; `23843` had an equal full cancellation, while one 74,215-unit line dominated `23166`.

Four filters, in order: (1) drop `C` invoices; (2) keep only `Quantity > 0` and `UnitPrice > 0`; (3) keep product-like `StockCode` values (five digits plus optional letters); (4) remove only positive lines matched to an earlier cancellation for the same customer, product, and exact quantity.
Waterfall: **541,909 -> 532,621 -> 530,104 -> 527,725 -> 524,894** rows (raw population through each filter).

Exact matching requires the same quantity and a known `CustomerID`, so partial returns and returns with missing `CustomerID` are not matched; their corresponding positive lines remain.

The 1,031 no-earlier-purchase cancellations total 18,053 returned units, about 0.34% of the 5,351,288 clean units, and some may refer to purchases before the file begins.